# Step 3: Querying with DuckDB — Schema-on-Read

At this stage, the lake consists solely of files organized into folders. DuckDB queries them directly using a glob pattern, without requiring an import step, a loading process, or a separate database to maintain:

```sql
SELECT ... FROM 'lake/verkauf/**/*.parquet'
```

This is **schema-on-read**: the schema resides within the Parquet files themselves and is interpreted only at query time. This contrasts with a data warehouse, where a table's schema is fixed before any data is loaded.

In [ ]:
import os
from pathlib import Path

while not (Path.cwd() / "requirements.txt").exists():
    os.chdir("..")
print("Working directory:", Path.cwd())

In [ ]:
import duckdb

con = duckdb.connect()

con.sql("""
    SELECT region, SUM(revenue) AS total_revenue
    FROM 'lake/verkauf/**/*.parquet'
    GROUP BY region
    ORDER BY total_revenue DESC
""").show()

## Predicate Pushdown: Filtering on the Partition Column

Filtering on `jahr = 2026` and examining the query plan with `EXPLAIN ANALYZE` reveals the number of Parquet files actually opened. Partitions outside the filter are never accessed.

In [ ]:
con.sql("""
    EXPLAIN ANALYZE
    SELECT region, SUM(revenue) AS total_revenue
    FROM 'lake/verkauf/**/*.parquet'
    WHERE jahr = 2026
    GROUP BY region
""").show()

Note the `Filters:` entry in the scan node and the number of rows read. Because `jahr` is a partition column encoded in the folder path, DuckDB is able to skip entire files rather than reading and subsequently discarding rows.

## Column Pruning

Selecting only two of the five columns causes DuckDB to read only those columns from disk. Because Parquet stores each column separately, unused columns are never opened.

In [ ]:
con.sql("""
    EXPLAIN ANALYZE
    SELECT product, revenue
    FROM 'lake/verkauf/**/*.parquet'
    WHERE jahr = 2026
""").show()

## Timing Comparison: CSV vs. Partitioned Parquet

The same logical query — total revenue for 2026 — is executed once against the raw CSV file (which has no `jahr` column and must therefore be scanned in full, comparing the date on every row) and once against the partitioned Parquet dataset.

In [ ]:
import time

t0 = time.perf_counter()
csv_result = con.sql("""
    SELECT SUM(revenue) AS total_revenue
    FROM read_csv_auto('lake/raw/sales.csv')
    WHERE date >= '2026-01-01' AND date < '2027-01-01'
""").fetchone()
csv_time = time.perf_counter() - t0

t0 = time.perf_counter()
parquet_result = con.sql("""
    SELECT SUM(revenue) AS total_revenue
    FROM 'lake/verkauf/**/*.parquet'
    WHERE jahr = 2026
""").fetchone()
parquet_time = time.perf_counter() - t0

print(f"CSV      : {csv_time:.3f}s  -> {csv_result}")
print(f"Parquet  : {parquet_time:.3f}s  -> {parquet_result}")
print(f"Speedup  : {csv_time / parquet_time:.1f}x")

**Summary of the two optimizations applied automatically:**

- **Predicate pushdown** — the `jahr = 2026` filter allowed DuckDB to skip Parquet files for other years without opening them.
- **Column pruning** — only the columns referenced in the `SELECT`/`WHERE` clauses were read from each file.

Neither optimization is possible with the CSV file, which has no partitions to skip and no columnar layout to prune.